# Классификация: IC50 выше медианы

Задача: предсказать, превышает ли IC50 медианное значение выборки (45.34 mM).

Сравним два подхода: на всех 210 признаках и после удаления константных и почти константных признаков (158 признаков).

## 1. Импорты

In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 2. Загрузка и подготовка

Готовим два набора признаков: полный и сокращённый без константных и почти константных.

In [2]:
raw_data = pd.read_excel("../data/dataset.xlsx")
TARGET_COLUMNS = ["IC50, mM", "CC50, mM", "SI"]

data = raw_data.iloc[:, 1:].copy()
data = data.drop_duplicates()

feature_columns = [col for col in data.columns if col not in TARGET_COLUMNS]
X_full = data[feature_columns]

# Сокращённый набор
constant_mask = X_full.nunique(dropna=False) <= 1

dominant_share = X_full.apply(
    lambda column: column.value_counts(
        dropna=False,
        normalize=True,
    ).iloc[0]
)

near_constant_mask = dominant_share >= 0.95

X_reduced = X_full.loc[
    :,
    ~(constant_mask | near_constant_mask),
]

threshold = data["IC50, mM"].median()
y = (data["IC50, mM"] > threshold).astype(int)

print(f"Полный набор: {X_full.shape[1]} признаков")
print(f"Сокращённый: {X_reduced.shape[1]} признаков")
print(f"Класс 0: {(y == 0).sum()}, класс 1: {(y == 1).sum()} ({y.mean():.1%} положительных)")

Полный набор: 210 признаков
Сокращённый: 158 признаков
Класс 0: 485, класс 1: 484 (49.9% положительных)


## 3. Разбиение

Используем стратифицированное групповое разбиение. Объекты с одинаковыми значениями молекулярных дескрипторов относятся к одной группе и не могут одновременно попасть в обучение и тест.

Для подбора гиперпараметров также используем групповую стратифицированную кросс-валидацию.

In [3]:
groups = pd.util.hash_pandas_object(X_full, index=False)

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X_full, y, groups=groups)
)

Xf_train = X_full.iloc[train_idx]
Xf_test = X_full.iloc[test_idx]

Xr_train = X_reduced.iloc[train_idx]
Xr_test = X_reduced.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

print(f"Обучение: {len(Xf_train)}, тест: {len(Xf_test)}")

Обучение: 775, тест: 194


## 4. Метрики

- Accuracy — доля правильных ответов. При равных классах случайное угадывание даст 0.5
- F1 — среднее гармоническое точности и полноты положительного класса. Метрика учитывает как пропущенные положительные объекты, так и ложные положительные предсказания
- ROC-AUC — площадь под ROC-кривой. Показывает, насколько хорошо модель разделяет классы независимо от порога. 0.5 — случайно, 1.0 — идеально

Гиперпараметры подбираем по F1. Для сравнения используем baseline, который относит все объекты к положительному классу

## 5. Логистическая регрессия

Линейный классификатор. Строит разделяющую гиперплоскость. Признаки масштабируем через StandardScaler, пропуски заполняем медианой.

Параметр "C" управляет силой регуляризации: чем меньше "C", тем сильнее штраф и проще модель. Подбираем 5 значений на обоих наборах признаков.

In [4]:
# Полный набор
lr_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(random_state=42, max_iter=5000))
lr_params = {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}

lr_search = GridSearchCV(lr_pipe, lr_params, cv=cv, scoring="f1", n_jobs=-1)
lr_search.fit(Xf_train, y_train, groups=groups_train)

lr_cv = pd.DataFrame({"C": lr_params["logisticregression__C"], "CV F1": lr_search.cv_results_["mean_test_score"]})
print("Полный набор (210 признаков)")
display(lr_cv)
print(f"Лучший C: {lr_search.best_params_['logisticregression__C']}, CV F1: {lr_search.best_score_:.4f}")

Полный набор (210 признаков)


,C,CV F1
0,0.01,0.673245
1,0.10,0.681372
2,1.00,0.691923
3,10.00,0.673923
4,100.00,0.664946


Лучший C: 1, CV F1: 0.6919


In [5]:
lr_pred = lr_search.predict(Xf_test)
lr_proba = lr_search.predict_proba(Xf_test)[:, 1]
lr_full = {"name": "LR (полный)", "acc": accuracy_score(y_test, lr_pred), "f1": f1_score(y_test, lr_pred), "roc": roc_auc_score(y_test, lr_proba)}
print(f"Test Accuracy: {lr_full['acc']:.3f}, Test F1: {lr_full['f1']:.3f}, ROC-AUC: {lr_full['roc']:.3f}")

Test Accuracy: 0.665, Test F1: 0.663, ROC-AUC: 0.735


In [6]:
# Сокращённый набор
lr_pipe2 = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(random_state=42, max_iter=5000))
lr_search2 = GridSearchCV(lr_pipe2, lr_params, cv=cv, scoring="f1", n_jobs=-1)
lr_search2.fit(Xr_train, y_train, groups=groups_train)

lr_cv2 = pd.DataFrame({"C": lr_params["logisticregression__C"], "CV F1": lr_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор (158 признаков)")
display(lr_cv2)
print(f"Лучший C: {lr_search2.best_params_['logisticregression__C']}, CV F1: {lr_search2.best_score_:.4f}")

Сокращённый набор (158 признаков)


,C,CV F1
0,0.01,0.669911
1,0.10,0.685596
2,1.00,0.679872
3,10.00,0.656215
4,100.00,0.648120


Лучший C: 0.1, CV F1: 0.6856


In [7]:
lr_pred2 = lr_search2.predict(Xr_test)
lr_proba2 = lr_search2.predict_proba(Xr_test)[:, 1]
lr_reduced = {"name": "LR (сокращённый)", "acc": accuracy_score(y_test, lr_pred2), "f1": f1_score(y_test, lr_pred2), "roc": roc_auc_score(y_test, lr_proba2)}
print(f"Test Accuracy: {lr_reduced['acc']:.3f}, Test F1: {lr_reduced['f1']:.3f}, ROC-AUC: {lr_reduced['roc']:.3f}")

Test Accuracy: 0.603, Test F1: 0.635, ROC-AUC: 0.634


На полном наборе логистическая регрессия получила F1 = 0.663, на сокращённом — 0.635. Удаление признаков ухудшило все три метрики.

## 6. K ближайших соседей

Голосование K ближайших соседей. StandardScaler критичен — без него признаки с большим размахом задавят остальные при подсчёте расстояния. Подбираем число соседей и тип взвешивания (uniform — все равны, distance — по близости).

In [8]:
# Полный набор
knn_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier())
knn_params = {"kneighborsclassifier__n_neighbors": [3, 5, 7, 10, 15, 20], "kneighborsclassifier__weights": ["uniform", "distance"]}

knn_search = GridSearchCV(knn_pipe, knn_params, cv=cv, scoring="f1", n_jobs=-1)
knn_search.fit(Xf_train, y_train, groups=groups_train)

knn_cv = pd.DataFrame({"n_neighbors": knn_search.cv_results_["param_kneighborsclassifier__n_neighbors"], "weights": knn_search.cv_results_["param_kneighborsclassifier__weights"], "CV F1": knn_search.cv_results_["mean_test_score"]})
print("Полный набор (210 признаков)")
display(knn_cv.sort_values("CV F1", ascending=False))
print(f"Лучшие: {knn_search.best_params_}, CV F1: {knn_search.best_score_:.4f}")

Полный набор (210 признаков)


,n_neighbors,weights,CV F1
0,3,uniform,0.685123
9,15,distance,0.682011
3,5,distance,0.677505
1,3,distance,0.677324
2,5,uniform,0.674653
11,20,distance,0.668055
7,10,distance,0.664761
8,15,uniform,0.664221
5,7,distance,0.662280
4,7,uniform,0.650635


Лучшие: {'kneighborsclassifier__n_neighbors': 3, 'kneighborsclassifier__weights': 'uniform'}, CV F1: 0.6851


In [9]:
knn_pred = knn_search.predict(Xf_test)
knn_proba = knn_search.predict_proba(Xf_test)[:, 1]
knn_full = {"name": "KNN (полный)", "acc": accuracy_score(y_test, knn_pred), "f1": f1_score(y_test, knn_pred), "roc": roc_auc_score(y_test, knn_proba)}
print(f"Test Accuracy: {knn_full['acc']:.3f}, Test F1: {knn_full['f1']:.3f}, ROC-AUC: {knn_full['roc']:.3f}")

Test Accuracy: 0.716, Test F1: 0.709, ROC-AUC: 0.753


In [10]:
# Сокращённый набор
knn_pipe2 = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier())
knn_search2 = GridSearchCV(knn_pipe2, knn_params, cv=cv, scoring="f1", n_jobs=-1)
knn_search2.fit(Xr_train, y_train, groups=groups_train)

knn_cv2 = pd.DataFrame({"n_neighbors": knn_search2.cv_results_["param_kneighborsclassifier__n_neighbors"], "weights": knn_search2.cv_results_["param_kneighborsclassifier__weights"], "CV F1": knn_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор (158 признаков)")
display(knn_cv2.sort_values("CV F1", ascending=False))
print(f"Лучшие: {knn_search2.best_params_}, CV F1: {knn_search2.best_score_:.4f}")

Сокращённый набор (158 признаков)


,n_neighbors,weights,CV F1
9,15,distance,0.706012
3,5,distance,0.690455
2,5,uniform,0.690239
7,10,distance,0.687682
4,7,uniform,0.686429
1,3,distance,0.683813
8,15,uniform,0.680641
5,7,distance,0.680218
0,3,uniform,0.680108
11,20,distance,0.678040


Лучшие: {'kneighborsclassifier__n_neighbors': 15, 'kneighborsclassifier__weights': 'distance'}, CV F1: 0.7060


In [11]:
knn_pred2 = knn_search2.predict(Xr_test)
knn_proba2 = knn_search2.predict_proba(Xr_test)[:, 1]
knn_reduced = {"name": "KNN (сокращённый)", "acc": accuracy_score(y_test, knn_pred2), "f1": f1_score(y_test, knn_pred2), "roc": roc_auc_score(y_test, knn_proba2)}
print(f"Test Accuracy: {knn_reduced['acc']:.3f}, Test F1: {knn_reduced['f1']:.3f}, ROC-AUC: {knn_reduced['roc']:.3f}")

Test Accuracy: 0.649, Test F1: 0.679, ROC-AUC: 0.712


KNN показал лучший результат на полном наборе: F1 = 0.709. После сокращения набора признаков F1 снизился до 0.679, поэтому удаление признаков не улучшило качество модели.

## 7. Случайный лес

Ансамбль деревьев. Не требует масштабирования. Расширенная сетка: 48 комбинаций (n_estimators, max_depth, min_samples_leaf, max_features), полный перебор через GridSearchCV на обоих наборах.

In [12]:
# Полный набор
rf_pipe = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42))
rf_params = {"randomforestclassifier__n_estimators": [100, 200, 300, 500], "randomforestclassifier__max_depth": [10, 20, 30, None], "randomforestclassifier__min_samples_leaf": [1, 3, 5], "randomforestclassifier__max_features": ["sqrt", "log2", None]}

rf_search = GridSearchCV(rf_pipe, rf_params, cv=cv, scoring="f1", n_jobs=-1)
rf_search.fit(Xf_train, y_train, groups=groups_train)

rf_cv = pd.DataFrame({"n_estimators": rf_search.cv_results_["param_randomforestclassifier__n_estimators"], "max_depth": rf_search.cv_results_["param_randomforestclassifier__max_depth"].astype(str), "min_samples_leaf": rf_search.cv_results_["param_randomforestclassifier__min_samples_leaf"], "max_features": rf_search.cv_results_["param_randomforestclassifier__max_features"].astype(str), "CV F1": rf_search.cv_results_["mean_test_score"]})
print("Полный набор (210 признаков)")
display(rf_cv.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {rf_search.best_params_}, CV F1: {rf_search.best_score_:.4f}")

Полный набор (210 признаков)


,n_estimators,max_depth,min_samples_leaf,max_features,CV F1
38,300,20,1,sqrt,0.717102
111,500,None,1,sqrt,0.716973
39,500,20,1,sqrt,0.716973
75,500,30,1,sqrt,0.716973
74,300,30,1,sqrt,0.715546
110,300,None,1,sqrt,0.715546
86,300,30,1,log2,0.713639
122,300,None,1,log2,0.713639
121,200,None,1,log2,0.711719
85,200,30,1,log2,0.711719


Лучшие: {'randomforestclassifier__max_depth': 20, 'randomforestclassifier__max_features': 'sqrt', 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 300}, CV F1: 0.7171


In [13]:
rf_pred = rf_search.predict(Xf_test)
rf_proba = rf_search.predict_proba(Xf_test)[:, 1]
rf_full = {"name": "RF (полный)", "acc": accuracy_score(y_test, rf_pred), "f1": f1_score(y_test, rf_pred), "roc": roc_auc_score(y_test, rf_proba)}
print(f"Test Accuracy: {rf_full['acc']:.3f}, Test F1: {rf_full['f1']:.3f}, ROC-AUC: {rf_full['roc']:.3f}")

Test Accuracy: 0.660, Test F1: 0.689, ROC-AUC: 0.754


In [14]:
# Сокращённый набор
rf_pipe2 = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42))
rf_search2 = GridSearchCV(rf_pipe2, rf_params, cv=cv, scoring="f1", n_jobs=-1)
rf_search2.fit(Xr_train, y_train, groups=groups_train)

rf_cv2 = pd.DataFrame({"n_estimators": rf_search2.cv_results_["param_randomforestclassifier__n_estimators"], "max_depth": rf_search2.cv_results_["param_randomforestclassifier__max_depth"].astype(str), "min_samples_leaf": rf_search2.cv_results_["param_randomforestclassifier__min_samples_leaf"], "max_features": rf_search2.cv_results_["param_randomforestclassifier__max_features"].astype(str), "CV F1": rf_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор (158 признаков)")
display(rf_cv2.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {rf_search2.best_params_}, CV F1: {rf_search2.best_score_:.4f}")

Сокращённый набор (158 признаков)


,n_estimators,max_depth,min_samples_leaf,max_features,CV F1
74,300,30,1,sqrt,0.717887
110,300,None,1,sqrt,0.717887
38,300,20,1,sqrt,0.717887
75,500,30,1,sqrt,0.714745
111,500,None,1,sqrt,0.714745
39,500,20,1,sqrt,0.713794
82,300,30,5,sqrt,0.712445
46,300,20,5,sqrt,0.712445
118,300,None,5,sqrt,0.712445
117,200,None,5,sqrt,0.711657


Лучшие: {'randomforestclassifier__max_depth': 20, 'randomforestclassifier__max_features': 'sqrt', 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 300}, CV F1: 0.7179


In [15]:
rf_pred2 = rf_search2.predict(Xr_test)
rf_proba2 = rf_search2.predict_proba(Xr_test)[:, 1]
rf_reduced = {"name": "RF (сокращённый)", "acc": accuracy_score(y_test, rf_pred2), "f1": f1_score(y_test, rf_pred2), "roc": roc_auc_score(y_test, rf_proba2)}
print(f"Test Accuracy: {rf_reduced['acc']:.3f}, Test F1: {rf_reduced['f1']:.3f}, ROC-AUC: {rf_reduced['roc']:.3f}")

Test Accuracy: 0.655, Test F1: 0.697, ROC-AUC: 0.741


Случайный лес получил F1 = 0.689 на полном наборе и 0.697 на сокращённом. Сокращение признаков немного улучшило F1, но ROC-AUC снизился с 0.754 до 0.741.

## 8. Градиентный бустинг

Последовательное построение деревьев, каждое исправляет ошибки предыдущих. RandomizedSearchCV по 20 комбинациям на обоих наборах.

In [16]:
# Полный набор
gb_pipe = make_pipeline(SimpleImputer(strategy="median"), GradientBoostingClassifier(random_state=42))
gb_params = {"gradientboostingclassifier__n_estimators": [100, 200, 300], "gradientboostingclassifier__max_depth": [3, 5, 7], "gradientboostingclassifier__learning_rate": [0.01, 0.05, 0.1]}

gb_search = RandomizedSearchCV(gb_pipe, gb_params, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42)
gb_search.fit(Xf_train, y_train, groups=groups_train)

gb_cv = pd.DataFrame({"n_estimators": gb_search.cv_results_["param_gradientboostingclassifier__n_estimators"], "max_depth": gb_search.cv_results_["param_gradientboostingclassifier__max_depth"], "learning_rate": gb_search.cv_results_["param_gradientboostingclassifier__learning_rate"], "CV F1": gb_search.cv_results_["mean_test_score"]})
print("Полный набор (210 признаков)")
display(gb_cv.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {gb_search.best_params_}, CV F1: {gb_search.best_score_:.4f}")

Полный набор (210 признаков)


,n_estimators,max_depth,learning_rate,CV F1
12,300,5,0.01,0.718286
8,100,5,0.05,0.715627
1,200,5,0.05,0.714296
14,100,7,0.05,0.709817
6,200,7,0.05,0.708020
3,100,5,0.10,0.704826
19,100,3,0.10,0.703739
17,200,7,0.10,0.700956
16,100,5,0.01,0.700881
15,200,5,0.10,0.700577


Лучшие: {'gradientboostingclassifier__n_estimators': 300, 'gradientboostingclassifier__max_depth': 5, 'gradientboostingclassifier__learning_rate': 0.01}, CV F1: 0.7183


In [17]:
gb_pred = gb_search.predict(Xf_test)
gb_proba = gb_search.predict_proba(Xf_test)[:, 1]
gb_full = {"name": "GBT (полный)", "acc": accuracy_score(y_test, gb_pred), "f1": f1_score(y_test, gb_pred), "roc": roc_auc_score(y_test, gb_proba)}
print(f"Test Accuracy: {gb_full['acc']:.3f}, Test F1: {gb_full['f1']:.3f}, ROC-AUC: {gb_full['roc']:.3f}")

Test Accuracy: 0.665, Test F1: 0.695, ROC-AUC: 0.755


In [18]:
# Сокращённый набор
gb_pipe2 = make_pipeline(SimpleImputer(strategy="median"), GradientBoostingClassifier(random_state=42))
gb_search2 = RandomizedSearchCV(gb_pipe2, gb_params, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42)
gb_search2.fit(Xr_train, y_train, groups=groups_train)

gb_cv2 = pd.DataFrame({"n_estimators": gb_search2.cv_results_["param_gradientboostingclassifier__n_estimators"], "max_depth": gb_search2.cv_results_["param_gradientboostingclassifier__max_depth"], "learning_rate": gb_search2.cv_results_["param_gradientboostingclassifier__learning_rate"], "CV F1": gb_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор (158 признаков)")
display(gb_cv2.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {gb_search2.best_params_}, CV F1: {gb_search2.best_score_:.4f}")

Сокращённый набор (158 признаков)


,n_estimators,max_depth,learning_rate,CV F1
19,100,3,0.10,0.717146
15,200,5,0.10,0.715220
1,200,5,0.05,0.712961
12,300,5,0.01,0.711231
18,300,5,0.10,0.711102
3,100,5,0.10,0.709898
14,100,7,0.05,0.708833
8,100,5,0.05,0.707938
6,200,7,0.05,0.705589
11,200,5,0.01,0.702048


Лучшие: {'gradientboostingclassifier__n_estimators': 100, 'gradientboostingclassifier__max_depth': 3, 'gradientboostingclassifier__learning_rate': 0.1}, CV F1: 0.7171


In [19]:
gb_pred2 = gb_search2.predict(Xr_test)
gb_proba2 = gb_search2.predict_proba(Xr_test)[:, 1]
gb_reduced = {"name": "GBT (сокращённый)", "acc": accuracy_score(y_test, gb_pred2), "f1": f1_score(y_test, gb_pred2), "roc": roc_auc_score(y_test, gb_proba2)}
print(f"Test Accuracy: {gb_reduced['acc']:.3f}, Test F1: {gb_reduced['f1']:.3f}, ROC-AUC: {gb_reduced['roc']:.3f}")

Test Accuracy: 0.660, Test F1: 0.683, ROC-AUC: 0.734


Градиентный бустинг получил F1 = 0.695 на полном наборе и 0.683 на сокращённом. Удаление признаков немного ухудшило качество модели.

## 9. Сравнение

### baseline

In [20]:
baseline_pred = np.ones(len(y_test), dtype=int)
baseline_proba = np.ones(len(y_test))

baseline = {
    "name": "Baseline (все 1)",
    "acc": accuracy_score(y_test, baseline_pred),
    "f1": f1_score(y_test, baseline_pred),
    "roc": roc_auc_score(y_test, baseline_proba),
}

print(
    f"Baseline | Accuracy: {baseline['acc']:.3f}, "
    f"F1: {baseline['f1']:.3f}, "
    f"ROC-AUC: {baseline['roc']:.3f}"
)

Baseline | Accuracy: 0.500, F1: 0.667, ROC-AUC: 0.500


Две таблицы — полный и сокращённый наборы.

In [21]:
res_full = pd.DataFrame(
    [baseline, lr_full, knn_full, rf_full, gb_full]
).sort_values("f1", ascending=False).reset_index(drop=True)

res_full.columns = ["Модель", "Accuracy", "F1", "ROC-AUC"]

print("210 признаков")
display(res_full.round(3))

res_reduced = pd.DataFrame(
    [baseline, lr_reduced, knn_reduced, rf_reduced, gb_reduced]
).sort_values("f1", ascending=False).reset_index(drop=True)

res_reduced.columns = ["Модель", "Accuracy", "F1", "ROC-AUC"]

print("158 признаков")
display(res_reduced.round(3))

210 признаков


,Модель,Accuracy,F1,ROC-AUC
0,KNN (полный),0.716,0.709,0.753
1,GBT (полный),0.665,0.695,0.755
2,RF (полный),0.660,0.689,0.754
3,Baseline (все 1),0.500,0.667,0.500
4,LR (полный),0.665,0.663,0.735


158 признаков


,Модель,Accuracy,F1,ROC-AUC
0,RF (сокращённый),0.655,0.697,0.741
1,GBT (сокращённый),0.660,0.683,0.734
2,KNN (сокращённый),0.649,0.679,0.712
3,Baseline (все 1),0.500,0.667,0.500
4,LR (сокращённый),0.603,0.635,0.634


## 10. Выводы

Лучший результат на тестовой выборке показал KNN на полном наборе признаков: F1 = 0.709, Accuracy = 0.716 и ROC-AUC = 0.753.

На сокращённом наборе лучший результат получил случайный лес с F1 = 0.697. Однако его ROC-AUC снизился с 0.754 до 0.741.

Baseline, который относит все объекты к положительному классу, получил F1 = 0.667. Поэтому преимущество лучшей модели по F1 составляет около 0.04. Логистическая регрессия на обоих наборах не превзошла baseline по F1.

Удаление константных и почти константных признаков не дало общего улучшения. Немного вырос F1 случайного леса, но ухудшились результаты логистической регрессии, KNN и градиентного бустинга.

Для итогового решения выбираем KNN на полном наборе признаков. При этом качество модели остаётся умеренным.

Групповое разбиение исключает попадание объектов с одинаковыми дескрипторами одновременно в обучение и тест.